In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("/workspaces/SebastianBot")

In [ ]:
from cloud.dependencies.clients import resolve_calendar_event_client

client = resolve_calendar_event_client()

In [3]:
from datetime import datetime, timezone

time_min = datetime.now(tz=timezone.utc).isoformat()

response = (
    client._service._service.events()
    .list(
        calendarId="oneironaut.oml@gmail.com",
        timeMin=time_min,
        q="014437370",
        singleEvents=True,
        orderBy="startTime",
    )
    .execute()
)

items = response.get("items", [])
print(f"Found {len(items)} event(s)")
for item in items:
    print(item)

Found 1 event(s)
{'kind': 'calendar#event', 'etag': '"3551878185972254"', 'id': 'ckom4d336oq6abb670q64b9k69h6cbb26oq34b9h6gsm4p9kcoom8dpm6o', 'status': 'confirmed', 'htmlLink': 'https://www.google.com/calendar/event?eid=Y2tvbTRkMzM2b3E2YWJiNjcwcTY0YjlrNjloNmNiYjI2b3EzNGI5aDZnc200cDlrY29vbThkcG02byBvbmVpcm9uYXV0Lm9tbEBt', 'created': '2026-04-11T20:24:52.000Z', 'updated': '2026-04-11T20:24:52.986Z', 'summary': 'Bibo', 'description': 'Id: 014437370', 'creator': {'email': 'oneironaut.oml@gmail.com', 'self': True}, 'organizer': {'email': 'oneironaut.oml@gmail.com', 'self': True}, 'start': {'dateTime': '2026-05-04T22:30:00+02:00', 'timeZone': 'Europe/Berlin'}, 'end': {'dateTime': '2026-05-04T23:30:00+02:00', 'timeZone': 'Europe/Berlin'}, 'iCalUID': 'ckom4d336oq6abb670q64b9k69h6cbb26oq34b9h6gsm4p9kcoom8dpm6o@google.com', 'sequence': 0, 'reminders': {'useDefault': True}, 'eventType': 'default'}


In [ ]:
from pydantic import BaseModel, ConfigDict


class EventDateTime(BaseModel):
    dateTime: str | None = None
    timeZone: str | None = None
    date: str | None = None

    model_config = ConfigDict(extra="allow")


class CalendarEventResponse(BaseModel):
    id: str
    summary: str | None = None
    description: str | None = None
    start: EventDateTime | None = None
    end: EventDateTime | None = None

    model_config = ConfigDict(extra="allow")


event_responses = [
    CalendarEventResponse.model_validate(item) for item in response.get("items", [])
]

In [11]:
event_responses

[CalendarEventResponse(id='ckom4d336oq6abb670q64b9k69h6cbb26oq34b9h6gsm4p9kcoom8dpm6o', summary='Bibo', description='Id: 014437370', start=EventDateTime(dateTime='2026-05-04T22:30:00+02:00', timeZone='Europe/Berlin', date=None), end=EventDateTime(dateTime='2026-05-04T23:30:00+02:00', timeZone='Europe/Berlin', date=None), kind='calendar#event', etag='"3551878185972254"', status='confirmed', htmlLink='https://www.google.com/calendar/event?eid=Y2tvbTRkMzM2b3E2YWJiNjcwcTY0YjlrNjloNmNiYjI2b3EzNGI5aDZnc200cDlrY29vbThkcG02byBvbmVpcm9uYXV0Lm9tbEBt', created='2026-04-11T20:24:52.000Z', updated='2026-04-11T20:24:52.986Z', creator={'email': 'oneironaut.oml@gmail.com', 'self': True}, organizer={'email': 'oneironaut.oml@gmail.com', 'self': True}, iCalUID='ckom4d336oq6abb670q64b9k69h6cbb26oq34b9h6gsm4p9kcoom8dpm6o@google.com', sequence=0, reminders={'useDefault': True}, eventType='default')]

In [12]:
from zoneinfo import ZoneInfo


def _to_datetime(value: EventDateTime | None) -> datetime | None:
    if value is None:
        return None
    if value.dateTime:
        return datetime.fromisoformat(value.dateTime)
    if value.date:
        tz = ZoneInfo(value.timeZone) if value.timeZone else timezone.utc
        return datetime.fromisoformat(value.date).replace(tzinfo=tz)
    return None


class CalendarEvent(BaseModel):
    id: str
    summary: str | None = None
    description: str | None = None
    start: datetime | None = None
    end: datetime | None = None

    model_config = ConfigDict(extra="allow")

    @classmethod
    def from_response(cls, event: CalendarEventResponse) -> "CalendarEvent":
        return cls(
            id=event.id,
            summary=event.summary,
            description=event.description,
            start=_to_datetime(event.start),
            end=_to_datetime(event.end),
        )


calendar_events = [CalendarEvent.from_response(event) for event in event_responses]
calendar_events

[CalendarEvent(id='ckom4d336oq6abb670q64b9k69h6cbb26oq34b9h6gsm4p9kcoom8dpm6o', summary='Bibo', description='Id: 014437370', start=datetime.datetime(2026, 5, 4, 22, 30, tzinfo=datetime.timezone(datetime.timedelta(seconds=7200))), end=datetime.datetime(2026, 5, 4, 23, 30, tzinfo=datetime.timezone(datetime.timedelta(seconds=7200))))]